In [12]:
import pandas as pd
from sklearn.datasets import make_classification

# Generate a unique synthetic fraud dataset matching the company metrics
X_raw, y_raw = make_classification(
    n_samples=40000,          # Similar size to the project scope
    n_features=30,            # 30 transaction features (like V1-V28, Time, Amount)
    n_clusters_per_class=1,
    weights=[0.9983, 0.0017], # Exact class imbalance required by the project
    flip_y=0,                 # Keeps the labels clean
    random_state=77           # Changing this number creates an entirely unique dataset
)

# Convert to a structured DataFrame
feature_names = [f'Transaction_Feature_{i}' for i in range(1, 31)]
df = pd.DataFrame(X_raw, columns=feature_names)
df['Target_Class'] = y_raw

print("Unique Dataset Generated! Shape:", df.shape)

Unique Dataset Generated! Shape: (40000, 31)


In [13]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['Target_Class'])
y = df['Target_Class']

# Stratified split to protect the evaluation data
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=77,
    stratify=y
)

In [14]:
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Pipeline 1: Linear Model Pipeline (Requires scaling)
linear_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('oversample', SMOTE(random_state=77)),
    ('classifier', LogisticRegression(max_iter=1000, random_state=77))
])

# Pipeline 2: Ensemble Tree Pipeline (Does not require scaling)
ensemble_pipeline = Pipeline([
    ('oversample', SMOTE(random_state=77)),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=77, n_jobs=-1))
])

In [15]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['Target_Class'])
y = df['Target_Class']

# Stratified split to protect the evaluation data
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=77,
    stratify=y
)

In [11]:
from sklearn.model_selection import GridSearchCV

# Define parameters to tune both SMOTE neighbors and tree depth together safely
param_tuning_grid = {
    'oversample__k_neighbors': [3, 5],
    'classifier__max_depth': [10, 20, None]
}

tuning_engine = GridSearchCV(
    estimator=ensemble_pipeline,
    param_grid=param_tuning_grid,
    scoring='roc_auc',
    cv=3,
    n_jobs=-1
)

# Train and optimize
tuning_engine.fit(X_train, y_train)
best_production_model = tuning_engine.best_estimator_

In [16]:
from sklearn.metrics import classification_report, roc_auc_score

# Predict on the untouched test data
predictions = best_production_model.predict(X_test)
probabilities = best_production_model.predict_proba(X_test)[:, 1]

print("--- FINAL PRODUCTION MODEL PERFORMANCE ---")
print(classification_report(y_test, predictions, target_names=['Legitimate', 'Fraudulent']))
print(f"Final ROC-AUC Score: {roc_auc_score(y_test, probabilities):.4f}")

--- FINAL PRODUCTION MODEL PERFORMANCE ---
              precision    recall  f1-score   support

  Legitimate       1.00      1.00      1.00      7986
  Fraudulent       1.00      1.00      1.00        14

    accuracy                           1.00      8000
   macro avg       1.00      1.00      1.00      8000
weighted avg       1.00      1.00      1.00      8000

Final ROC-AUC Score: 1.0000
